# Computational efficiency

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from matplotlib.ticker import ScalarFormatter

In [ ]:
SCRIPT_DIR = Path('.')
REPO_ROOT = SCRIPT_DIR / '../../../'
RESULTS_DIR = (REPO_ROOT / 'results').resolve()
TIME_BENCHMARKS_DIR = RESULTS_DIR / 'time_benchmarks'

KNOWN_TOP_LEVEL_METHODS = (
    'tmvec1', 'tmvec2', 'tmvec2_student',
    'diamond', 'foldseek', 'foldseek_gpu', 'prostt5_gpu',
)


def load_runtime_pair(method, encoding_path, query_path, source):
    encoding = pd.read_csv(encoding_path)[['encoding_size', 'mean_seconds']].copy()
    encoding['method'] = method
    encoding['source'] = source
    query_df = pd.read_csv(query_path).copy()
    query_value_column = 'total_mean' if 'total_mean' in query_df.columns else 'search_mean'
    query = query_df[['query_size', 'database_size', query_value_column]].rename(
        columns={query_value_column: 'total_mean'}
    )
    query['method'] = method
    query['source'] = source
    return encoding, query


def method_name_from_config(config_path, default_label=None):
    config = json.loads(config_path.read_text())
    comparison_mode = config.get('comparison_mode')
    if comparison_mode:
        return comparison_mode
    if default_label is not None:
        return default_label
    dirname = config_path.parent.name
    if 'prostt5' in dirname or 'prostt5_weights' in config:
        return f"prostt5_{'gpu' if config.get('use_gpu') else 'cpu'}"
    if 'foldseek' in dirname or 'foldseek_binary' in config:
        if 'exhaustive' in dirname:
            return 'foldseek_gpu_exhaustive' if config.get('use_gpu') else 'foldseek_cpu_exhaustive'
        return 'foldseek_gpu' if config.get('use_gpu') else 'foldseek'
    return dirname


def discover_top_level_runs():
    seen_methods = set()
    for config_path in sorted(RESULTS_DIR.glob('*_benchmark_config.json')):
        suffix = '_benchmark_config.json'
        method = config_path.name[:-len(suffix)] if config_path.name.endswith(suffix) else config_path.stem
        enc_path = RESULTS_DIR / f'{method}_encoding_times.csv'
        qry_path = RESULTS_DIR / f'{method}_query_times.csv'
        if not enc_path.exists() or not qry_path.exists():
            continue
        seen_methods.add(method)
        yield method_name_from_config(config_path, default_label=method), enc_path, qry_path, str(config_path)
    for method in KNOWN_TOP_LEVEL_METHODS:
        if method in seen_methods:
            continue
        enc_path = RESULTS_DIR / f'{method}_encoding_times.csv'
        qry_path = RESULTS_DIR / f'{method}_query_times.csv'
        if not enc_path.exists() or not qry_path.exists():
            continue
        yield method, enc_path, qry_path, str(RESULTS_DIR)


def discover_runtime_runs():
    if not TIME_BENCHMARKS_DIR.exists():
        return
    for config_path in sorted(TIME_BENCHMARKS_DIR.glob('*/benchmark_config.json')):
        enc_path = config_path.parent / 'encoding_times.csv'
        qry_path = config_path.parent / 'query_times.csv'
        if not enc_path.exists() or not qry_path.exists():
            continue
        yield method_name_from_config(config_path), enc_path, qry_path, str(config_path.parent)


encoding_tables, query_tables = [], []
for method, enc_path, qry_path, source in [*discover_top_level_runs(), *discover_runtime_runs()]:
    enc, qry = load_runtime_pair(method, enc_path, qry_path, source)
    encoding_tables.append(enc)
    query_tables.append(qry)

if not encoding_tables or not query_tables:
    raise FileNotFoundError('No runtime benchmark outputs were found under results/')

encoding_df = pd.concat(encoding_tables, axis=0, ignore_index=True)
query_df = pd.concat(query_tables, axis=0, ignore_index=True)

In [3]:
plt.rcParams['svg.fonttype'] = 'none'

In [4]:
methods = ['tmvec1', 'tmvec2', 'tmvec2_student', 'foldseek', 'foldseek_gpu', 'prostt5_gpu']
names = ['TM-Vec', 'TM-Vec 2', 'TM-Vec 2s', 'Foldseek', 'Foldseek GPU', 'Foldseek PLM']

Encoding

In [ ]:
df = encoding_df.copy()
df.head()

In [ ]:
sizes = sorted(df['encoding_size'].unique())
sizes

In [ ]:
plt.figure(figsize=(5, 4))
for method, name in zip(methods, names):
    df_ = df.query(f'method == "{method}"')
    plt.plot('encoding_size', 'mean_seconds', data=df_, marker='o', label=name)
plt.legend(title='Method')
plt.xscale('log')
plt.yscale('log')
plt.gca().xaxis.set_major_formatter(ScalarFormatter())
plt.gca().yaxis.set_major_formatter(ScalarFormatter())
plt.xlabel('Number of sequences (log)')
plt.ylabel('Runtime (sec) (log)')
plt.title('Encoding')
plt.tight_layout()
plt.savefig('plots/encoding.svg')

In [8]:
df['seqs_per_second'] = df['encoding_size'] / df['mean_seconds']

In [ ]:
s_max = df.groupby('method')['seqs_per_second'].max().loc[methods]
s_max

In [ ]:
plt.figure(figsize=(4, 4))
plt.bar(x=methods, height=s_max.to_numpy())
for i, method in enumerate(methods):
    plt.text(i, s_max[method], '{:.4g}'.format(s_max[method]), ha='center')
plt.yscale('log')
plt.ylim(top=plt.ylim()[1] * 2)
plt.xlabel('Method')
plt.xticks(range(4), names)
plt.ylabel('Max. sequences per second')
plt.gca().yaxis.set_major_formatter(ScalarFormatter())
plt.title('Encoding')
plt.savefig('plots/maxseqs.svg')

Query

In [ ]:
df = query_df.copy()
df.head()

Benchmark results of BLAST and DIAMOND were adopted from Figure S7 of the TM-Vec 1 paper.

- https://www.nature.com/articles/s41587-023-01917-2

In [12]:
blast = pd.DataFrame([
    [10, 1000, 9],
    [100, 1000, 16],
    [1000, 1000, 65],
    [10, 10000, 10],
    [100, 10000, 25],
    [1000, 10000, 131],
    [10, 100000, 14],
    [100, 100000, 63],
    [1000, 100000, 175]
], columns=['query_size', 'database_size', 'total_mean']).assign(method='blast')

In [13]:
diamond = pd.DataFrame([
    [10, 1000, 0.521],
    [100, 1000, 0.512],
    [1000, 1000, 0.597],
    [10, 10000, 0.573],
    [100, 10000, 0.644],
    [1000, 10000, 0.775],
    [10, 100000, 1.234],
    [100, 100000, 1.395],
    [1000, 100000, 1.743]
], columns=['query_size', 'database_size', 'total_mean']).assign(method='diamond')

In [14]:
df = pd.concat([df, blast, diamond])

In [ ]:
db_sizes = sorted(df['database_size'].unique())
db_sizes

In [16]:
methods = ['tmvec1', 'tmvec2', 'tmvec2_student', 'foldseek', 'foldseek_gpu', 'prostt5_gpu']
names = ['TM-Vec', 'TM-Vec 2', 'TM-Vec 2s', 'Foldseek', 'Foldseek GPU', 'Foldseek PLM']

In [ ]:
fig, axes = plt.subplots(1, 3, sharey=True, figsize=(8.5, 4))
for i, size in enumerate(db_sizes):
    ax = axes[i]
    for method, name in zip(methods, names):
        df_ = df.query(f'database_size == {size} & method == "{method}"')
        ax.plot('query_size', 'total_mean', data=df_, marker='o', label=name)
    ax.set_xscale('log')
    ax.xaxis.set_major_formatter(ScalarFormatter())
    if i == 0:
        ax.set_ylabel('Runtime (sec) (log)')
        ax.set_yscale('log')
        ax.yaxis.set_major_formatter(ScalarFormatter())
    if i == 1:
        ax.set_xlabel('Number of query sequences (log)')
    ax.set_title(f'Database size: {int(size / 1000)}k')
    if i == 2:
        ax.legend(title='Method', loc='center left', bbox_to_anchor=(1.0, 0.5))
fig.tight_layout()
fig.savefig('plots/query.svg')